# Privacy-aware joint DNN model deployment and partitioning optimization for collaborative edge inference services

### sample

In [2]:
import numpy as np
import pandas as pd
from itertools import product


# =========================================================
# 1. Build client/cloud profiles from your measured arrays
# =========================================================

# =========================================================
# SAMPLE DATA (replace later with your real measurements)
# =========================================================

# Example:
# Model has 5 layers
# Each number = inference time of that layer (seconds)

DEVICE_A_4CORE = np.array([
    0.010,
    0.015,
    0.020,
    0.025,
    0.030,
], dtype=float)

DEVICE_B_2CORE = np.array([
    0.020,
    0.030,
    0.040,
    0.050,
    0.060,
], dtype=float)

DEVICE_C_1CORE = np.array([
    0.040,
    0.050,
    0.070,
    0.090,
    0.120,
], dtype=float)

# Cloud is much faster
DEVICE_CLOUD = np.array([
    0.003,
    0.004,
    0.005,
    0.006,
    0.008,
], dtype=float)

# Raw input image size (MB)
RAW_INPUT_MB = 5.0

# Activation size after each cut point
# Since model has 5 layers -> need K-1 = 4 values
CUT_DATA_SIZES_MB = np.array([
    3.5,
    2.0,
    1.0,
    0.4,
], dtype=float)

def build_client_layer_time(client_types):
    """
    client_types example:
    ["A", "A", "B", "B", "C", "C", "C", "A", "B", "C"]
    """
    return np.vstack([DEVICE_PROFILES[t] for t in client_types]).astype(float)


def build_cloud_layer_time(num_clouds, cloud_scales=None):
    """
    cloud_scales:
    - None: all cloud servers use DEVICE_CLOUD
    - [1.0, 1.2, 0.8]&#58; cloud 1 normal, cloud 2 slower, cloud 3 faster
      larger scale means slower inference.
    """
    if cloud_scales is None:
        cloud_scales = np.ones(num_clouds)

    cloud_scales = np.asarray(cloud_scales, dtype=float)
    assert len(cloud_scales) == num_clouds

    return cloud_scales[:, None] * DEVICE_CLOUD[None, :]


def build_activation_sizes(raw_input_mb, cut_data_sizes_mb, num_layers):
    """
    Convention:
    z = 0       : full cloud, transmit raw input
    z = 1       : client runs layer 0, transmit output after layer 0
    ...
    z = K - 1   : client runs layers 0 ... K-2
    z = K       : full local, transmit nothing

    Your CUT_DATA_SIZES_MB has length K-1, so:
    activation_size[z=0] = RAW_INPUT_MB
    activation_size[z=1 ... K-1] = CUT_DATA_SIZES_MB
    activation_size[z=K] = 0
    """
    cut_data_sizes_mb = np.asarray(cut_data_sizes_mb, dtype=float)

    if len(cut_data_sizes_mb) == num_layers - 1:
        return np.concatenate([[raw_input_mb], cut_data_sizes_mb, [0.0]])

    if len(cut_data_sizes_mb) == num_layers + 1:
        return cut_data_sizes_mb

    raise ValueError(
        f"CUT_DATA_SIZES_MB length must be K-1 or K+1. "
        f"Got {len(cut_data_sizes_mb)}, while K={num_layers}."
    )


# =========================================================
# 2. Core latency model
# =========================================================

def prefix_client_times(client_layer_time):
    """
    prefix[i, z] = time for client i to run layers [0, ..., z-1]
    """
    n_clients, num_layers = client_layer_time.shape
    prefix = np.zeros((n_clients, num_layers + 1), dtype=float)
    prefix[:, 1:] = np.cumsum(client_layer_time, axis=1)
    return prefix


def suffix_cloud_times(cloud_layer_time):
    """
    suffix[j, z] = time for cloud j to run layers [z, ..., K-1]
    """
    n_clouds, num_layers = cloud_layer_time.shape
    suffix = np.zeros((n_clouds, num_layers + 1), dtype=float)
    suffix[:, :num_layers] = np.cumsum(cloud_layer_time[:, ::-1], axis=1)[:, ::-1]
    suffix[:, num_layers] = 0.0
    return suffix


def evaluate_assignment(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
    assignment,
    objective="round_time",
    latency_mode="sequential",
    cloud_contention=True,
    share_bandwidth=True,
    fixed_cut=None,
):
    """
    assignment[i] = cloud server selected by client i.

    objective:
    - "round_time": minimize max latency among clients. Good for FPS.
    - "mean_latency": minimize average latency.

    latency_mode:
    - "sequential": T = T_client + T_comm + T_cloud
    - "pipeline":   T = max(T_client, T_comm, T_cloud)

    cloud_contention:
    - True: if many clients use the same cloud, cloud time is multiplied by cloud load.
    - False: ideal parallel cloud execution.

    share_bandwidth:
    - True: bandwidth is divided by the number of clients assigned to the same cloud.
    - False: bandwidth_MBps[i, j] is treated as already measured effective bandwidth.
    """
    client_layer_time = np.asarray(client_layer_time, dtype=float)
    cloud_layer_time = np.asarray(cloud_layer_time, dtype=float)
    activation_mb = np.asarray(activation_mb, dtype=float)
    bandwidth_MBps = np.asarray(bandwidth_MBps, dtype=float)
    assignment = np.asarray(assignment, dtype=int)

    n_clients, num_layers = client_layer_time.shape
    n_clouds, num_layers_2 = cloud_layer_time.shape

    assert num_layers == num_layers_2
    assert activation_mb.shape == (num_layers + 1,)
    assert bandwidth_MBps.shape == (n_clients, n_clouds)
    assert assignment.shape == (n_clients,)

    client_prefix = prefix_client_times(client_layer_time)
    cloud_suffix = suffix_cloud_times(cloud_layer_time)

    loads = np.bincount(assignment, minlength=n_clouds)

    best_cuts = np.zeros(n_clients, dtype=int)
    latencies = np.zeros(n_clients, dtype=float)

    for i in range(n_clients):
        j = assignment[i]

        candidate_cuts = [fixed_cut] if fixed_cut is not None else range(num_layers + 1)

        best_latency = float("inf")
        best_cut = None

        for z in candidate_cuts:
            t_client = client_prefix[i, z]

            if z == num_layers:
                # Full local inference
                t_comm = 0.0
                t_cloud = 0.0
            else:
                effective_bw = bandwidth_MBps[i, j]

                if share_bandwidth:
                    effective_bw = effective_bw / max(loads[j], 1)

                t_comm = activation_mb[z] / max(effective_bw, 1e-12)

                t_cloud = cloud_suffix[j, z]

                if cloud_contention:
                    t_cloud = t_cloud * max(loads[j], 1)

            if latency_mode == "pipeline":
                total_latency = max(t_client, t_comm, t_cloud)
            else:
                total_latency = t_client + t_comm + t_cloud

            if total_latency < best_latency:
                best_latency = total_latency
                best_cut = z

        best_cuts[i] = best_cut
        latencies[i] = best_latency

    round_time = float(np.max(latencies))
    mean_latency = float(np.mean(latencies))

    if objective == "round_time":
        obj = round_time
    elif objective == "mean_latency":
        obj = mean_latency
    else:
        raise ValueError("objective must be 'round_time' or 'mean_latency'.")

    system_fps = n_clients / round_time if round_time > 0 else float("inf")
    mean_client_fps = float(np.mean(1.0 / np.maximum(latencies, 1e-12)))

    return {
        "objective": obj,
        "assignment": assignment.copy(),
        "cuts": best_cuts,
        "latencies": latencies,
        "loads": loads,
        "mean_latency": mean_latency,
        "p95_latency": float(np.percentile(latencies, 95)),
        "round_time": round_time,
        "system_fps": system_fps,
        "mean_client_fps": mean_client_fps,
    }


# =========================================================
# 3. Baselines
# =========================================================

def full_local_baseline(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
):
    n_clients, num_layers = client_layer_time.shape
    assignment = np.zeros(n_clients, dtype=int)

    return evaluate_assignment(
        client_layer_time,
        cloud_layer_time,
        activation_mb,
        bandwidth_MBps,
        assignment,
        fixed_cut=num_layers,
        objective="round_time",
        cloud_contention=False,
        share_bandwidth=False,
    )


def exact_assignment_baseline(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
    objective="round_time",
    latency_mode="sequential",
    cloud_contention=True,
    share_bandwidth=True,
    fixed_cut=None,
    max_states=200_000,
):
    """
    Exact search over all client-cloud assignments.
    For 10 clients and 3 cloud servers: 3^10 = 59049 states, still feasible.
    """
    n_clients = client_layer_time.shape[0]
    n_clouds = cloud_layer_time.shape[0]

    total_states = n_clouds ** n_clients

    if total_states > max_states:
        raise ValueError(
            f"Exact search has {total_states} states. "
            f"Use coalition_baseline instead."
        )

    best = None

    for assign_tuple in product(range(n_clouds), repeat=n_clients):
        assignment = np.array(assign_tuple, dtype=int)

        result = evaluate_assignment(
            client_layer_time,
            cloud_layer_time,
            activation_mb,
            bandwidth_MBps,
            assignment,
            objective=objective,
            latency_mode=latency_mode,
            cloud_contention=cloud_contention,
            share_bandwidth=share_bandwidth,
            fixed_cut=fixed_cut,
        )

        if best is None or result["objective"] < best["objective"]:
            best = result

    return best


def full_cloud_baseline(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
    objective="round_time",
    latency_mode="sequential",
    cloud_contention=True,
    share_bandwidth=True,
):
    """
    Full cloud means cut z = 0.
    Client sends raw input to cloud, cloud runs the whole model.
    """
    return exact_assignment_baseline(
        client_layer_time,
        cloud_layer_time,
        activation_mb,
        bandwidth_MBps,
        objective=objective,
        latency_mode=latency_mode,
        cloud_contention=cloud_contention,
        share_bandwidth=share_bandwidth,
        fixed_cut=0,
    )


def init_assignment_by_best_link(bandwidth_MBps):
    """
    Simple initialization: each client selects cloud with highest bandwidth.
    """
    return np.argmax(bandwidth_MBps, axis=1).astype(int)


def coalition_baseline(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
    objective="round_time",
    latency_mode="sequential",
    cloud_contention=True,
    share_bandwidth=True,
    max_iter=100,
    try_exchange=True,
    seed=0,
):
    """
    Coalition-style baseline:
    - Start from best-link assignment.
    - Try moving one client to another cloud.
    - Accept if objective improves.
    - Optionally try swapping two clients between different clouds.
    """
    rng = np.random.default_rng(seed)

    n_clients = client_layer_time.shape[0]
    n_clouds = cloud_layer_time.shape[0]

    assignment = init_assignment_by_best_link(bandwidth_MBps)

    best = evaluate_assignment(
        client_layer_time,
        cloud_layer_time,
        activation_mb,
        bandwidth_MBps,
        assignment,
        objective=objective,
        latency_mode=latency_mode,
        cloud_contention=cloud_contention,
        share_bandwidth=share_bandwidth,
    )

    for _ in range(max_iter):
        improved = False

        # Switch operation
        for i in rng.permutation(n_clients):
            current_cloud = assignment[i]

            local_best = best
            local_best_assignment = assignment.copy()

            for new_cloud in range(n_clouds):
                if new_cloud == current_cloud:
                    continue

                candidate_assignment = assignment.copy()
                candidate_assignment[i] = new_cloud

                result = evaluate_assignment(
                    client_layer_time,
                    cloud_layer_time,
                    activation_mb,
                    bandwidth_MBps,
                    candidate_assignment,
                    objective=objective,
                    latency_mode=latency_mode,
                    cloud_contention=cloud_contention,
                    share_bandwidth=share_bandwidth,
                )

                if result["objective"] < local_best["objective"] - 1e-12:
                    local_best = result
                    local_best_assignment = candidate_assignment

            if local_best["objective"] < best["objective"] - 1e-12:
                assignment = local_best_assignment
                best = local_best
                improved = True

        # Exchange operation
        if try_exchange:
            pairs = [(a, b) for a in range(n_clients) for b in range(a + 1, n_clients)]
            rng.shuffle(pairs)

            for a, b in pairs:
                if assignment[a] == assignment[b]:
                    continue

                candidate_assignment = assignment.copy()
                candidate_assignment[a], candidate_assignment[b] = (
                    candidate_assignment[b],
                    candidate_assignment[a],
                )

                result = evaluate_assignment(
                    client_layer_time,
                    cloud_layer_time,
                    activation_mb,
                    bandwidth_MBps,
                    candidate_assignment,
                    objective=objective,
                    latency_mode=latency_mode,
                    cloud_contention=cloud_contention,
                    share_bandwidth=share_bandwidth,
                )

                if result["objective"] < best["objective"] - 1e-12:
                    assignment = candidate_assignment
                    best = result
                    improved = True
                    break

        if not improved:
            break

    return best


# =========================================================
# 4. Report helper
# =========================================================

def result_to_row(name, result):
    return {
        "method": name,
        "mean_latency_s": result["mean_latency"],
        "p95_latency_s": result["p95_latency"],
        "round_time_s": result["round_time"],
        "system_fps": result["system_fps"],
        "mean_client_fps": result["mean_client_fps"],
        "loads": result["loads"].tolist(),
        "assignment": result["assignment"].tolist(),
        "cuts": result["cuts"].tolist(),
    }


def print_result(name, result):
    print(f"\n{name}")
    print("-" * len(name))
    print("assignment:", result["assignment"])
    print("cuts      :", result["cuts"])
    print("loads     :", result["loads"])
    print(f"mean latency   : {result['mean_latency']:.6f} s")
    print(f"p95 latency    : {result['p95_latency']:.6f} s")
    print(f"round time     : {result['round_time']:.6f} s")
    print(f"system FPS     : {result['system_fps']:.2f}")
    print(f"mean client FPS: {result['mean_client_fps']:.2f}")

# =========================================================
# 5. Example experiment: 10 clients, 3 cloud servers
# =========================================================

client_types = [
    "A", "A", "A",     # 3 clients using device A
    "B", "B", "B",     # 3 clients using device B
    "C", "C", "C", "C" # 4 clients using device C
]

num_clouds = 3

client_layer_time = build_client_layer_time(client_types)

# If all cloud servers are identical:
cloud_layer_time = build_cloud_layer_time(num_clouds)

# If cloud servers have different speed, use this instead:
# cloud_layer_time = build_cloud_layer_time(num_clouds, cloud_scales=[1.0, 1.2, 0.85])

num_layers = client_layer_time.shape[1]

activation_mb = build_activation_sizes(
    raw_input_mb=RAW_INPUT_MB,
    cut_data_sizes_mb=CUT_DATA_SIZES_MB,
    num_layers=num_layers,
)

# Bandwidth matrix [num_clients, num_clouds], unit: MB/s.
# Replace this with your measured network bandwidth.
bandwidth_MBps = np.array([
    [80,  60,  50],
    [75,  65,  55],
    [70,  80,  60],
    [60,  90,  70],
    [55,  85,  75],
    [50,  70,  95],
    [40,  60, 100],
    [45,  75,  90],
    [65,  55,  85],
    [70,  50,  80],
], dtype=float)

# Recommended for FPS-oriented evaluation:
objective = "round_time"

# Use "sequential" first. Then try "pipeline" if your system supports pipeline execution.
latency_mode = "sequential"

# If bandwidth_MBps is already measured as effective per-client bandwidth, set share_bandwidth=False.
# If bandwidth is total shared bandwidth per cloud, set share_bandwidth=True.
share_bandwidth = True

# If cloud can process clients fully in parallel, set cloud_contention=False.
# If cloud compute is shared/serialized among clients, keep True.
cloud_contention = True


res_local = full_local_baseline(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
)

res_cloud = full_cloud_baseline(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
    objective=objective,
    latency_mode=latency_mode,
    cloud_contention=cloud_contention,
    share_bandwidth=share_bandwidth,
)

res_exact = exact_assignment_baseline(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
    objective=objective,
    latency_mode=latency_mode,
    cloud_contention=cloud_contention,
    share_bandwidth=share_bandwidth,
)

res_coalition = coalition_baseline(
    client_layer_time,
    cloud_layer_time,
    activation_mb,
    bandwidth_MBps,
    objective=objective,
    latency_mode=latency_mode,
    cloud_contention=cloud_contention,
    share_bandwidth=share_bandwidth,
    max_iter=100,
    try_exchange=True,
    seed=1,
)

print_result("Full Local", res_local)
print_result("Full Cloud", res_cloud)
print_result("Exact Assignment + Exhaustive Cut", res_exact)
print_result("Coalition Baseline + Exhaustive Cut", res_coalition)

df = pd.DataFrame([
    result_to_row("Full Local", res_local),
    result_to_row("Full Cloud", res_cloud),
    result_to_row("Exact Assignment + Exhaustive Cut", res_exact),
    result_to_row("Coalition Baseline + Exhaustive Cut", res_coalition),
])

NameError: name 'DEVICE_PROFILES' is not defined

### Optimizer

In [ ]:
import json
import numpy as np
import pandas as pd

# =========================================================
# 1. ĐỌC DỮ LIỆU TRỰC TIẾP TỪ CÁC FILE JSON CỦA BẠN
# =========================================================
def load_simulation_data():
    try:
        with open('ES.json', 'r') as f:
            es_data = json.load(f)
        with open('MD.json', 'r') as f:
            md_data = json.load(f)
        with open('yolo11n_dag_profile.json', 'r') as f:
            dag_profile = json.load(f)
        return es_data, md_data, dag_profile
    except FileNotFoundError as e:
        print(f"LỖI HỆ THỐNG: Vui lòng đảm bảo các file JSON nằm cùng thư mục chạy! Đang giả lập dữ liệu...")
        # Dự phòng nếu chạy trên môi trường chưa upload file
        return None, None, None

EDGE_SERVERS_DATA, MOBILE_DEVICES_DATA, YOLO11N_DAG_PROFILE = load_simulation_data()

# Tính toán các hằng số kích thước hệ thống
N_CLIENTS = len(MOBILE_DEVICES_DATA)
N_CLOUDS = len(EDGE_SERVERS_DATA)

# Tự động vá điểm cắt cuối cùng (Layer 23 - Detect) bị thiếu trong file JSON của bạn
if len(YOLO11N_DAG_PROFILE) == 23:
    YOLO11N_DAG_PROFILE.append({
        "cut_point": 23,
        "name": "Layer_23_Detect",
        "flops_giga": 0.0500,
        "payload_size_mb": 0.0050,
        "crossing_tensors": [23]
    })

TOTAL_LAYERS = len(YOLO11N_DAG_PROFILE)

# =========================================================
# 2. CHUYỂN ĐỔI SANG MA TRẬN TOÁN HỌC ĐỘNG DỰA TRÊN FILE JSON
# =========================================================
def build_latency_matrices():
    flops_array = np.array([layer['flops_giga'] for layer in YOLO11N_DAG_PROFILE])
    payload_array = np.array([layer['payload_size_mb'] for layer in YOLO11N_DAG_PROFILE])

    md_gflops = np.array([md['compute_capacity_gflops'] for md in MOBILE_DEVICES_DATA])
    es_gflops = np.array([es['compute_capacity_gflops'] for es in EDGE_SERVERS_DATA])

    # client_layer_time[i, z] = Thời gian chạy tĩnh của riêng layer z trên MD i
    client_layer_time = flops_array[None, :] / md_gflops[:, None]
    # cloud_layer_time[j, z] = Thời gian chạy tĩnh của riêng layer z trên ES j
    cloud_layer_time = flops_array[None, :] / es_gflops[:, None]

    # Thiết lập ma trận băng thông không dây hiệu dụng (Đơn vị: MB/s)
    bandwidth_MBps = np.zeros((N_CLIENTS, N_CLOUDS), dtype=float)
    for i, md in enumerate(MOBILE_DEVICES_DATA):
        for j, es in enumerate(EDGE_SERVERS_DATA):
            # Lấy min giữa Uplink của thiết bị di động và tổng dung lượng trạm biên (Mbps) / 8.0 -> MB/s
            bandwidth_MBps[i, j] = min(md['uplink_mbps'], es['bandwidth_mbps']) / 8.0

    return client_layer_time, cloud_layer_time, payload_array, bandwidth_MBps

# =========================================================
# 3. CORE LATENCY MODEL (ĐÃ PHÂN TÁCH INDEX ĐIỂM CẮT)
# =========================================================
def prefix_client_times(client_layer_time):
    """ Thời gian tích lũy MD tự chạy từ đầu mạng đến hết lớp z """
    prefix = np.zeros_like(client_layer_time)
    prefix[:, 1:] = np.cumsum(client_layer_time[:, 1:], axis=1)
    return prefix

def suffix_cloud_times(cloud_layer_time):
    """ Thời gian tích lũy Edge Server phải gánh từ lớp z+1 đến hết mạng """
    suffix = np.zeros_like(cloud_layer_time)
    suffix[:, :-1] = np.cumsum(cloud_layer_time[:, :0:-1], axis=1)[:, ::-1]
    return suffix

def evaluate_assignment(
    client_layer_time, cloud_layer_time, activation_mb, bandwidth_MBps,
    assignment, objective="mean_latency", cloud_contention=True, share_bandwidth=True, fixed_cut=None
):
    client_prefix = prefix_client_times(client_layer_time)
    cloud_suffix = suffix_cloud_times(cloud_layer_time)

    # Đếm số lượng MD trong từng cụm (Mẫu số N_m(t) trong bài báo)
    loads = np.bincount(assignment, minlength=N_CLOUDS)

    best_cuts = np.zeros(N_CLIENTS, dtype=int)
    latencies = np.zeros(N_CLIENTS, dtype=float)

    for i in range(N_CLIENTS):
        j = assignment[i]
        best_latency = float("inf")
        best_cut = 0

        # Nếu cấu hình chạy Baseline cố định điểm cắt, ngược lại quét cạn từ 0 đến 23
        candidate_cuts = [fixed_cut] if fixed_cut is not None else range(TOTAL_LAYERS)

        for z in candidate_cuts:
            t_client = client_prefix[i, z]

            if z == TOTAL_LAYERS - 1: # Điểm cắt cuối cùng => Ép chạy Full Local
                t_comm = 0.0
                t_cloud = 0.0
            else:
                effective_bw = bandwidth_MBps[i, j]
                # Công thức bài báo: Băng thông bị chia sẻ đều
                if share_bandwidth:
                    effective_bw = effective_bw / max(loads[j], 1)

                t_comm = activation_mb[z] / max(effective_bw, 1e-12)
                t_cloud = cloud_suffix[j, z]

                # Công thức bài báo: Sức mạnh tính toán bị chia đều => Thời gian tăng lên tỷ lệ thuận với số tải
                if cloud_contention:
                    t_cloud = t_cloud * max(loads[j], 1)

            total_latency = t_client + t_comm + t_cloud

            if total_latency < best_latency:
                best_latency = total_latency
                best_cut = z

        best_cuts[i] = best_cut
        latencies[i] = best_latency

    round_time = float(np.max(latencies))
    mean_latency = float(np.mean(latencies))
    obj = mean_latency if objective == "mean_latency" else round_time

    return {
        "objective": obj,
        "assignment": assignment.copy(),
        "cuts": best_cuts,
        "latencies": latencies,
        "loads": loads,
        "mean_latency": mean_latency,
        "round_time": round_time,
    }

# =========================================================
# 4. COALITION FORMATION GAME (THUẬT TOÁN BÀI BÁO)
# =========================================================
def init_assignment_by_best_link(bandwidth_MBps):
    return np.argmax(bandwidth_MBps, axis=1).astype(int)

def coalition_baseline(
    client_layer_time, cloud_layer_time, activation_mb, bandwidth_MBps,
    objective="mean_latency", max_iter=100, try_exchange=True, seed=42
):
    rng = np.random.default_rng(seed)
    assignment = init_assignment_by_best_link(bandwidth_MBps)

    best = evaluate_assignment(
        client_layer_time, cloud_layer_time, activation_mb, bandwidth_MBps, assignment, objective
    )

    for iteration in range(max_iter):
        improved = False

        # Switch Operation (MD nhảy cụm Server)
        for i in rng.permutation(N_CLIENTS):
            current_cloud = assignment[i]
            local_best = best
            local_best_assignment = assignment.copy()

            for new_cloud in range(N_CLOUDS):
                if new_cloud == current_cloud: continue

                candidate_assignment = assignment.copy()
                candidate_assignment[i] = new_cloud

                result = evaluate_assignment(
                    client_layer_time, cloud_layer_time, activation_mb,
                    bandwidth_MBps, candidate_assignment, objective
                )

                if result["objective"] < local_best["objective"] - 1e-12:
                    local_best = result
                    local_best_assignment = candidate_assignment

            if local_best["objective"] < best["objective"] - 1e-12:
                assignment = local_best_assignment
                best = local_best
                improved = True

        # Exchange Operation (2 MD hoán đổi Server cho nhau)
        if try_exchange:
            pairs = [(a, b) for a in range(N_CLIENTS) for b in range(a + 1, N_CLIENTS)]
            rng.shuffle(pairs)

            for a, b in pairs:
                if assignment[a] == assignment[b]: continue

                candidate_assignment = assignment.copy()
                candidate_assignment[a], candidate_assignment[b] = candidate_assignment[b], candidate_assignment[a]

                result = evaluate_assignment(
                    client_layer_time, cloud_layer_time, activation_mb,
                    bandwidth_MBps, candidate_assignment, objective
                )

                if result["objective"] < best["objective"] - 1e-12:
                    assignment = candidate_assignment
                    best = result
                    improved = True
                    break

        if not improved:
            print(f"[ĐIỀU PHỐI] Cấu trúc phân cụm đạt trạng thái Cân bằng Nash sau {iteration+1} vòng lặp.")
            break

    return best

# =========================================================
# 5. THỰC THI VÀ TRÍCH XUẤT THỐNG KÊ CHI TIẾT
# =========================================================
if __name__ == "__main__":
    client_time, cloud_time, act_mb, bandwidth_matrix = build_latency_matrices()

    # 1. Khởi chạy Baseline: Full Local (Chạy 100% trên thiết bị di động)
    assign_local = np.zeros(N_CLIENTS, dtype=int)
    res_local = evaluate_assignment(
        client_time, cloud_time, act_mb, bandwidth_matrix, assign_local,
        cloud_contention=False, share_bandwidth=False, fixed_cut=TOTAL_LAYERS-1
    )

    # 2. Khởi chạy Thuật toán Đề xuất: Coalition Game
    res_coalition = coalition_baseline(
        client_time, cloud_time, act_mb, bandwidth_matrix, objective="mean_latency"
    )

    # Đọc mảng tên trực quan từ danh sách cấu trúc profile
    yolo_layer_names = [layer['name'] for layer in YOLO11N_DAG_PROFILE]

    print("\n" + "="*25 + " PHÂN PHỐI CHI TIẾT THEO FILE JSON CỦA BẠN " + "="*25)
    for i in range(N_CLIENTS):
        md_name = MOBILE_DEVICES_DATA[i]["md_id"]
        es_idx = res_coalition['assignment'][i]
        es_name = EDGE_SERVERS_DATA[es_idx]["es_id"]
        cut_idx = res_coalition['cuts'][i]
        latency = res_coalition['latencies'][i]
        payload = act_mb[cut_idx]

        print(f"-> {md_name:<6} | Gán vào: {es_name:<4} | Điểm cắt: Quy định tại lớp {cut_idx:>2} ({yolo_layer_names[cut_idx]:<22}) | Payload truyền mạng: {payload:>5.2f} MB | Tổng Trễ: {latency:.4f}s")

    print("="*92)
    print(f" Tải trọng phân bổ tại trạm biên : ES_1: {res_coalition['loads'][0]} máy | ES_2: {res_coalition['loads'][1]} máy | ES_3: {res_coalition['loads'][2]} máy")
    print(f" Độ trễ trung bình hệ thống đề xuất: {res_coalition['mean_latency']:.4f} giây")
    print(f" Độ trễ lớn nhất hệ thống đề xuất  : {res_coalition['round_time']:.4f} giây")
    print("="*92)

    # Xuất bảng đối chiếu thống kê sang cấu trúc DataFrame của Pandas
    df_report = pd.DataFrame([
        {"Phương pháp so sánh": "Full Local (Baseline)", "Độ trễ trung bình (s)": res_local["mean_latency"], "Độ trễ lớn nhất (s)": res_local["round_time"]},
        {"Phương pháp so sánh": "Trò chơi Liên minh (Đề xuất)", "Độ trễ trung bình (s)": res_coalition["mean_latency"], "Độ trễ lớn nhất (s)": res_coalition["round_time"]}
    ])
    try:
        from IPython.display import display
        display(df_report)
    except ImportError:
        print(df_report)

[ĐIỀU PHỐI] Cấu trúc phân cụm đạt trạng thái Cân bằng Nash sau 1 vòng lặp.

========================= PHÂN PHỐI CHI TIẾT THEO FILE JSON CỦA BẠN =========================
-> MD_1   | Gán vào: ES_1 | Điểm cắt: Quy định tại lớp 23 (Layer_23_Detect       ) | Payload truyền mạng:  0.01 MB | Tổng Trễ: 0.0376s
-> MD_2   | Gán vào: ES_1 | Điểm cắt: Quy định tại lớp 23 (Layer_23_Detect       ) | Payload truyền mạng:  0.01 MB | Tổng Trễ: 0.0704s
-> MD_3   | Gán vào: ES_1 | Điểm cắt: Quy định tại lớp 23 (Layer_23_Detect       ) | Payload truyền mạng:  0.01 MB | Tổng Trễ: 0.0282s
-> MD_4   | Gán vào: ES_1 | Điểm cắt: Quy định tại lớp 23 (Layer_23_Detect       ) | Payload truyền mạng:  0.01 MB | Tổng Trễ: 0.1127s
-> MD_5   | Gán vào: ES_1 | Điểm cắt: Quy định tại lớp 23 (Layer_23_Detect       ) | Payload truyền mạng:  0.01 MB | Tổng Trễ: 0.0188s
-> MD_6   | Gán vào: ES_1 | Điểm cắt: Quy định tại lớp 23 (Layer_23_Detect       ) | Payload truyền mạng:  0.01 MB | Tổng Trễ: 0.0470s
-> MD_7   | Gán vào:

,Phương pháp so sánh,Độ trễ trung bình (s),Độ trễ lớn nhất (s)
0,Full Local (Baseline),0.04359,0.112714
1,Trò chơi Liên minh (Đề xuất),0.04359,0.112714


### GPT Optimizer

In [ ]:
import json
import numpy as np
from itertools import product

# =========================================================
# 1. LOAD JSON FILES
# =========================================================

with open("MD.json", "r") as f:
    MDs = json.load(f)

with open("ES.json", "r") as f:
    ESs = json.load(f)

with open("yolo11n_dag_profile.json", "r") as f:
    DAG = json.load(f)

# =========================================================
# 2. BUILD DAG INFORMATION
# =========================================================

cut_points = [x["cut_point"] for x in DAG]

# FLOPs executed at each layer
layer_flops = np.array([
    x["flops_giga"] for x in DAG
])

# activation upload size after each cut
activation_mb = np.array([
    x["payload_size_mb"] for x in DAG
])

K = len(layer_flops)

# cumulative local FLOPs
prefix_flops = np.zeros(K + 1)

prefix_flops[1:] = np.cumsum(layer_flops)

# remaining cloud FLOPs
suffix_flops = np.zeros(K + 1)

suffix_flops[:K] = np.cumsum(
    layer_flops[::-1]
)[::-1]

# =========================================================
# 3. BUILD CLIENT / SERVER PROFILES
# =========================================================

n_clients = len(MDs)
n_servers = len(ESs)

client_gflops = np.array([
    md["compute_capacity_gflops"]
    for md in MDs
])

server_gflops = np.array([
    es["compute_capacity_gflops"]
    for es in ESs
])

server_bandwidth = np.array([
    es["bandwidth_mbps"]
    for es in ESs
])

uplink_bandwidth = np.array([
    md["uplink_mbps"]
    for md in MDs
])

# =========================================================
# 4. BANDWIDTH MATRIX
# =========================================================

# effective upload bandwidth:
# min(MD uplink, ES total bw)

bandwidth_MBps = np.zeros(
    (n_clients, n_servers)
)

for i in range(n_clients):

    for j in range(n_servers):

        mbps = min(
            uplink_bandwidth[i],
            server_bandwidth[j]
        )

        # Mbps -> MB/s
        bandwidth_MBps[i, j] = mbps / 8.0

# =========================================================
# 5. LATENCY EVALUATION
# =========================================================

def evaluate_assignment(
    assignment,
    objective="mean_latency",
    pipeline=True,
):

    loads = np.bincount(
        assignment,
        minlength=n_servers
    )

    best_cuts = np.zeros(n_clients, dtype=int)

    latencies = np.zeros(n_clients)

    for i in range(n_clients):

        j = assignment[i]

        best_latency = 1e18
        best_cut = 0

        for z in range(K + 1):

            # ====================================
            # LOCAL COMPUTE
            # ====================================

            local_flops = prefix_flops[z]

            t_local = (
                local_flops
                / client_gflops[i]
            )

            # ====================================
            # FULL LOCAL
            # ====================================

            if z == K:

                total_latency = t_local

            else:

                # ====================================
                # SHARED BANDWIDTH
                # ====================================

                bw = (
                    bandwidth_MBps[i, j]
                    / max(loads[j], 1)
                )

                # ====================================
                # UPLOAD
                # ====================================

                t_upload = (
                    activation_mb[z]
                    / max(bw, 1e-9)
                )

                # ====================================
                # EDGE COMPUTE
                # ====================================

                edge_flops = suffix_flops[z]

                effective_server = (
                    server_gflops[j]
                    / max(loads[j], 1)
                )

                t_edge = (
                    edge_flops
                    / effective_server
                )

                # ====================================
                # PIPELINE LATENCY
                # ====================================

                if pipeline:

                    total_latency = max(
                        t_local,
                        max(t_upload, t_edge)
                    )

                else:

                    total_latency = (
                        t_local
                        + t_upload
                        + t_edge
                    )

            # ====================================
            # BEST CUT
            # ====================================

            if total_latency < best_latency:

                best_latency = total_latency
                best_cut = z

        best_cuts[i] = best_cut
        latencies[i] = best_latency

    mean_latency = np.mean(latencies)

    round_time = np.max(latencies)

    if objective == "mean_latency":
        obj = mean_latency
    else:
        obj = round_time

    system_fps = n_clients / round_time

    return {
        "objective": obj,
        "assignment": assignment,
        "cuts": best_cuts,
        "latencies": latencies,
        "loads": loads,
        "mean_latency": mean_latency,
        "round_time": round_time,
        "system_fps": system_fps,
    }

# =========================================================
# 6. EXACT SEARCH
# =========================================================

def exact_search():

    best = None

    total_states = (
        n_servers ** n_clients
    )

    print("TOTAL STATES =", total_states)

    for assign_tuple in product(
        range(n_servers),
        repeat=n_clients
    ):

        assignment = np.array(assign_tuple)

        result = evaluate_assignment(
            assignment
        )

        if (
            best is None
            or result["objective"]
            < best["objective"]
        ):

            best = result

    return best

# =========================================================
# 7. COALITION SEARCH
# =========================================================

def coalition_search(max_iter=100):

    # init by strongest link

    assignment = np.argmax(
        bandwidth_MBps,
        axis=1
    )

    best = evaluate_assignment(
        assignment
    )

    for _ in range(max_iter):

        improved = False

        # ====================================
        # SWITCH OPERATION
        # ====================================

        for i in range(n_clients):

            current = assignment[i]

            for new_server in range(n_servers):

                if new_server == current:
                    continue

                candidate = assignment.copy()

                candidate[i] = new_server

                result = evaluate_assignment(
                    candidate
                )

                if (
                    result["objective"]
                    < best["objective"]
                ):

                    assignment = candidate
                    best = result
                    improved = True

        # ====================================
        # EXCHANGE OPERATION
        # ====================================

        for a in range(n_clients):

            for b in range(a + 1, n_clients):

                if assignment[a] == assignment[b]:
                    continue

                candidate = assignment.copy()

                candidate[a], candidate[b] = (
                    candidate[b],
                    candidate[a],
                )

                result = evaluate_assignment(
                    candidate
                )

                if (
                    result["objective"]
                    < best["objective"]
                ):

                    assignment = candidate
                    best = result
                    improved = True

        if not improved:
            break

    return best

# =========================================================
# 8. RUN
# =========================================================

# WARNING:
# 3^9 = 19683 states
# exact search still feasible

res_exact = exact_search()

res_coalition = coalition_search()

# =========================================================
# 9. PRINT RESULTS
# =========================================================

def print_result(name, result):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("\nAssignment:")

    for i, a in enumerate(result["assignment"]):

        print(
            f"{MDs[i]['md_id']} "
            f"-> "
            f"{ESs[a]['es_id']}"
        )

    print("\nCut Points:")

    for i, z in enumerate(result["cuts"]):

        cut_name = (
            "FULL_LOCAL"
            if z == K
            else DAG[z]["name"]
        )

        print(
            f"{MDs[i]['md_id']} "
            f"cut={z} "
            f"({cut_name})"
        )

    print("\nServer Loads:")
    print(result["loads"])

    print("\nMean Latency:")
    print(result["mean_latency"])

    print("\nRound Time:")
    print(result["round_time"])

    print("\nSystem FPS:")
    print(result["system_fps"])

print_result(
    "EXACT SEARCH RESULT",
    res_exact
)

print_result(
    "COALITION SEARCH RESULT",
    res_coalition
)

TOTAL STATES = 19683

EXACT SEARCH RESULT

Assignment:
MD_1 -> ES_1
MD_2 -> ES_1
MD_3 -> ES_2
MD_4 -> ES_3
MD_5 -> ES_2
MD_6 -> ES_3
MD_7 -> ES_1
MD_8 -> ES_2
MD_9 -> ES_2

Cut Points:
MD_1 cut=4 (Layer_4_Conv_P3/8)
MD_2 cut=4 (Layer_4_Conv_P3/8)
MD_3 cut=4 (Layer_4_Conv_P3/8)
MD_4 cut=4 (Layer_4_Conv_P3/8)
MD_5 cut=4 (Layer_4_Conv_P3/8)
MD_6 cut=4 (Layer_4_Conv_P3/8)
MD_7 cut=4 (Layer_4_Conv_P3/8)
MD_8 cut=23 (FULL_LOCAL)
MD_9 cut=23 (FULL_LOCAL)

Server Loads:
[3 4 2]

Mean Latency:
0.4650766798941799

Round Time:
0.6896

System FPS:
13.051044083526682

COALITION SEARCH RESULT

Assignment:
MD_1 -> ES_1
MD_2 -> ES_1
MD_3 -> ES_2
MD_4 -> ES_3
MD_5 -> ES_2
MD_6 -> ES_3
MD_7 -> ES_1
MD_8 -> ES_2
MD_9 -> ES_2

Cut Points:
MD_1 cut=4 (Layer_4_Conv_P3/8)
MD_2 cut=4 (Layer_4_Conv_P3/8)
MD_3 cut=4 (Layer_4_Conv_P3/8)
MD_4 cut=4 (Layer_4_Conv_P3/8)
MD_5 cut=4 (Layer_4_Conv_P3/8)
MD_6 cut=4 (Layer_4_Conv_P3/8)
MD_7 cut=4 (Layer_4_Conv_P3/8)
MD_8 cut=23 (FULL_LOCAL)
MD_9 cut=23 (FULL_LOCAL)

Ser